# Model A vs. Model B: Is the Gap Real?

You ran both models on your eval set. Model A scored 78%, Model B scored 75%.
Is that a real difference? With N=50, the 95% CI on that 3-point gap often spans zero.

This notebook walks through the complete statistical workflow:
from raw scores to a defensible conclusion.

In [7]:
import warnings
import numpy as np
import promptstats as pstats

warnings.filterwarnings("ignore")  # suppress internal pipeline warnings
rng = np.random.default_rng(42)

## The Setup

We evaluate **two models on the same 50 prompts** (a *paired* design).
Each item gets a binary score: 1 if the model's response passes, 0 if it fails.
We're using synthetic data here, but you can swap these scores with your own eval results.

The paired structure is key: because each prompt is seen by both models,
we can compute a per-item *difference* and bootstrap that directly.
This is more statistically powerful than treating the two samples as independent.

In [8]:
N = 50

# Synthetic binary scores (1 = pass, 0 = fail)
# True pass rates: Model A 0.70, Model B 0.65 — a 5-point gap in reality
scores_a = rng.binomial(1, 0.70, N).astype(float)
scores_b = rng.binomial(1, 0.65, N).astype(float)

mean_a = scores_a.mean()
mean_b = scores_b.mean()
obs_diff = mean_a - mean_b

print(f"Model A:  {mean_a:.1%}  ({int(scores_a.sum())}/{N} passed)")
print(f"Model B:  {mean_b:.1%}  ({int(scores_b.sum())}/{N} passed)")
print(f"Gap:     {obs_diff:+.1%}")

Model A:  64.0%  (32/50 passed)
Model B:  72.0%  (36/50 passed)
Gap:     -8.0%


## Computing a CI on the Difference

The observed gap is a *sample statistic* — it estimates the true gap but carries
sampling uncertainty. We use `pstats.compare_models()` to get bootstrapped confidence
intervals on each model's mean *and* on the gap between them.

Internally, the function treats the aligned score arrays as a paired design (each row
is the same prompt seen by both models) and bootstraps per-item differences.
We use `method="auto"`, which looks at the setup and input data types and tries to 
deduce the most appropriate statistical test to run for the comparison.

In [9]:
report = pstats.compare_models(
    {"Model A": scores_a, "Model B": scores_b},
    method="auto",
    rng=np.random.default_rng(0),
)

# Per-model statistics
for label, stats in report.model_stats.items():
    print(f"{label}: mean={stats.mean:.1%}  95% CI [{stats.ci_low:.1%}, {stats.ci_high:.1%}]")

print()

# Gap between the two models
pair = report.pairwise.get("Model A", "Model B")
print(f"Gap (A − B):  {pair.point_diff:+.1%}")
print(f"95% CI:       [{pair.ci_low:+.1%}, {pair.ci_high:+.1%}]")
print(f"p-value:      {pair.p_value:.3f}")
print()

print(report.quick_summary())

Model A: mean=64.0%  95% CI [49.1%, 78.8%]
Model B: mean=72.0%  95% CI [57.9%, 85.3%]

Gap (A − B):  -8.0%
95% CI:       [-22.5%, +9.1%]
p-value:      0.503

No significant difference between 'Model B' and 'Model A' (Δmean=+0.080, 95% CI [-0.091, 0.225], p=0.5034, holm-corrected, newcombe)


Note that "Newcombe" is named as the test. Since we set `method="auto"`, `promptstats` looked at our data and deduced it was binary (all 0 or 1s). Since we're comparing two models (a pairwise comparison), we use the Newcombe method for calculating confidence intervals, which adapts Wilson score intervals for the pairwise case. `promptstats` uses Newcombe as our simulations confirmed it was the most reliable statistical method for this data type. 

## What 'Statistically Distinguishable' Means

A CI that spans zero does **not** mean the two models are equal.
It means your data is **insufficient to rule out** the possibility that they are.
The gap could be real — you just don't have enough samples to confirm it.

The honest report is:
> *"Model A scored X% vs. Model B's Y% on our 50-item eval.
  The 95% CI on the gap is [lo, hi], which includes zero.
  We cannot conclude that Model A is better at this sample size."*

The wrong report is: *"Model A outperformed Model B (78% vs 75%)"* — full stop,
no uncertainty, no sample size, implying a definitive conclusion.

## How Much Data Do You Actually Need?

If the CI spans zero, the next question is: *how large would N need to be*
to reliably detect the gap you're seeing?

We estimate this with a simple simulation: for a range of N values,
we repeatedly sample from models with the observed means and check what
fraction of CIs exclude zero (= empirical power).

In [14]:
def empirical_power(p_a, p_b, n, n_sims=100, alpha=0.05, seed=1):
    """Fraction of simulated trials where the gap is declared significant."""
    rng_sim = np.random.default_rng(seed)
    detected = 0
    for _ in range(n_sims):
        a = rng_sim.binomial(1, p_a, n).astype(float)
        b = rng_sim.binomial(1, p_b, n).astype(float)
        rep = pstats.compare_models(
            {"Model A": a, "Model B": b},
            method="auto",
            alpha=alpha,
            rng=rng_sim,
        )
        if rep.significant:
            detected += 1
    return detected / n_sims


# True pass rates from our synthetic setup
p_a, p_b = 0.70, 0.65  # 5pp true gap

print(f"Power to detect {p_a - p_b:.0%} gap (binary scores, 95% CI):\n")
for n in [50, 100, 200, 400]:
    power = empirical_power(p_a, p_b, n)
    bar = "█" * int(power * 30)
    print(f"  N={n:4d}  {power:.0%}  {bar}")

Power to detect 5% gap (binary scores, 95% CI):

  N=  50  7%  ██
  N= 100  13%  ███
  N= 200  16%  ████
  N= 400  29%  ████████


## Summary

| What you have | What you can say |
|---|---|
| CI excludes zero, both bounds positive | Model A is reliably better |
| CI spans zero | Too close to call at this N |
| CI excludes zero, both bounds negative | Model B is reliably better |

**Key takeaways:**

- For binary scores, detecting a 5pp gap requires **N ≈ 200–400** per model at 80–95% power.
  Most evals use N=50–100, which can only reliably detect gaps of 10–15pp or larger.
- A paired design (same prompts for both models) is more powerful than independent samples.
  Use it whenever possible.
- Report CIs, not just means. A mean difference without a CI is an incomplete result.

Next: [Finding Your Best Prompt](./best-prompt.html) — extends this to k>2 comparisons
with multiple-comparison correction.